# 학습 및 검증 절차

ViT-B 출력 증류와 ViT-S 지도 미세조정을 동일한 실행 단위로 구성한다. 데이터 및 가중치 경로와 SHA-256은 `local_paths.json`에 지정한다. 기본 설정에서는 실험 구성을 조회한다. `LOAD_PRIVATE_INPUTS=True`에서 로컬 입력 로딩을 활성화하며, 한 epoch 학습은 별도의 `RUN_ONE_EPOCH` 설정으로 제어한다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository'
sys.path.insert(0, str(ROOT / 'src'))
import torch
torch.set_num_threads(2)
torch.manual_seed(20261112)
print('Repository', ROOT.name, '| PyTorch', torch.__version__)

## 1. 실험 설정

In [ ]:
import json
RECIPE = 'vits-supervised'  # or 'vitb-distillation'
DEVICE = 'cpu'             # GPU run: 'cuda'
LOAD_PRIVATE_INPUTS = False
RUN_ONE_EPOCH = False
config = json.loads((ROOT / f'configs/{RECIPE}.json').read_text())
print(config)

## 2. 데이터 분할 및 이미지·라벨 대응 검사

In [ ]:
import hashlib
from cpdino_lab.data import discover_splits, make_batch, read_local_paths
if LOAD_PRIVATE_INPUTS:
    paths = read_local_paths(ROOT / 'local_paths.json')
    splits = discover_splits(paths['dataset_root'])
    manifest = []
    for split in ['train', 'valid']:
        for sample in splits[split]:
            for path in [sample.image, sample.flow]:
                with path.open('rb') as stream:
                    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
                manifest.append((split, path.name, digest))
    dataset_hash = hashlib.sha256(json.dumps(manifest).encode()).hexdigest()
    print({key:len(value) for key,value in splits.items()})
else:
    print('Set local_paths.json and LOAD_PRIVATE_INPUTS to inspect private data')

## 3. 모델 구성 및 가중치 초기화

In [ ]:
from cpdino_lab.models import build_model
from cpdino_lab.weights import load_weights
from cpdino_lab.checkpoints import save_checkpoint, restore_checkpoint
if LOAD_PRIVATE_INPUTS:
    student = build_model(config['student'])
    key = 'vits_encoder' if config['student']=='vits' else 'vitb_model'
    provenance = load_weights(student, paths[key]['path'], paths[key]['sha256'], scope=config['initialization_scope'])
    student.to(DEVICE)
    teacher = None
    if config['teacher']:
        teacher = build_model(config['teacher'])
        load_weights(teacher, paths['vitl_model']['path'], paths['vitl_model']['sha256'], scope='model')
        teacher.to(DEVICE).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad],
        lr=config['learning_rate'], weight_decay=config['weight_decay'])
    identity = {'recipe':config, 'student':provenance, 'dataset_sha256':dataset_hash,
                'teacher_sha256': paths['vitl_model']['sha256'] if teacher is not None else None}
    run_dir = ROOT / 'runs' / config['name']
    last_epoch = 0
    if (run_dir / 'last.pt').exists():
        last_epoch = restore_checkpoint(run_dir / 'last.pt', student, optimizer, identity)
    print('Ready', config['name'], 'at epoch', last_epoch)

## 4. 단일 batch 학습 연산 검증

첫 번째 학습 이미지로 파라미터 갱신을 수행한다. 점검 이전 상태를 저장하고 이후 복원하여 본 학습의 초기 상태를 유지한다.

In [ ]:
from cpdino_lab.engine import train_step, evaluate
if LOAD_PRIVATE_INPUTS:
    save_checkpoint(run_dir/'inspection.pt', student, optimizer, last_epoch, identity)
    try:
        x, y = make_batch(splits['train'], [0], seed=42, device=DEVICE, tile_size=config['tile_size'])
        loss = train_step(student, optimizer, x, y, teacher, config['teacher_weight'])
        print('Batch shape', tuple(x.shape), '| loss', loss)
    finally:
        restore_checkpoint(run_dir/'inspection.pt', student, optimizer, identity)

## 5. Epoch 단위 학습, 검증 및 저장

`RUN_ONE_EPOCH=True`에서 이 셀을 실행하면 `last_epoch` 다음의 한 epoch를 수행한다. 반복 실행 시 저장된 epoch 이후부터 진행한다. 실험 설정의 목표 학습 길이는 500 epochs이다. 검증은 첫 epoch와 이후 10 epochs마다 수행한다. 현재 셀은 `last.pt`를 저장하며 validation 기반 `best.pt` 자동 선정은 포함하지 않는다.

In [ ]:
import numpy as np
if LOAD_PRIVATE_INPUTS and RUN_ONE_EPOCH:
    epoch = last_epoch + 1
    assert epoch <= config['epochs'], 'Recipe epoch budget already reached'
    order = np.random.default_rng(config['seed'] + epoch).permutation(len(splits['train']))
    weighted_loss = 0.0
    for offset in range(0, len(order), config['batch_size']):
        indices = order[offset:offset + config['batch_size']]
        x, y = make_batch(splits['train'], indices, epoch*1000+offset, DEVICE, config['tile_size'])
        value = train_step(student, optimizer, x, y, teacher, config['teacher_weight'])
        weighted_loss += value * len(indices)
    validation = None
    if epoch == 1 or epoch % config['validation_interval'] == 0:
        batches = (make_batch(splits['valid'], np.arange(offset, min(offset+config['batch_size'],len(splits['valid']))),
                   42+offset, DEVICE, config['tile_size'])
                   for offset in range(0, len(splits['valid']), config['batch_size']))
        validation = evaluate(student, batches)
    save_checkpoint(run_dir/'last.pt', student, optimizer, epoch, identity)
    last_epoch = epoch
    print({'epoch':epoch, 'train_loss':weighted_loss/len(order), 'validation_loss':validation})
else:
    print('One-epoch execution is controlled by LOAD_PRIVATE_INPUTS and RUN_ONE_EPOCH')

## 6. 재개 형식과 실험 기록

학습 재개는 공개 모듈의 optimizer 및 난수 상태 저장 형식을 사용한다. 원본 `best.pt`와 raw state dict의 모델 가중치는 `load_weights(scope="model")`로 로드할 수 있다. 기존 전체 학습 결과는 `03_results.ipynb`에 수록한다. 공개 모듈을 이용한 재학습 결과는 별도의 실행 기록으로 관리한다.